<a href="https://colab.research.google.com/github/VOIDxIIO/Colab-with-gui/blob/main/colab-with-gui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
2#@title 🚀 One-Click Colab Desktop (RustDesk + Chrome + XFCE)
#@markdown Turn Google Colab into a full Linux desktop you can control from any device.
#@markdown ---
#@markdown **Set your RustDesk password:**
RUSTDESK_PASSWORD = "colab1234" #@param {type:"string"}
#@markdown **Install Google Chrome?**
INSTALL_CHROME = True #@param {type:"boolean"}
#@markdown ---

import subprocess, os, time, IPython
from google.colab import output

def run(cmd, quiet=False):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not quiet and p.stdout.strip():
        print(p.stdout.strip())
    if not quiet and p.stderr.strip():
        print("STDERR:", p.stderr.strip())
    return p.returncode

def bg(cmd, log, env=None):
    subprocess.Popen(cmd, shell=True,
        stdout=open(log, "w"), stderr=subprocess.STDOUT,
        env={**os.environ, **(env or {})},
        start_new_session=True)

# 1. base packages
print("📦 [1/6] Installing base packages (~2 min)...")
run("apt-get update -qq", quiet=True)
run("apt-get install -y -qq dbus-x11 xvfb xfce4 xfce4-goodies wget curl", quiet=True)

# 2. RustDesk
print("🦀 [2/6] Installing RustDesk...")
run("wget -q https://github.com/rustdesk/rustdesk/releases/download/1.5.0/rustdesk-1.5.0-x86_64.deb -O /content/rustdesk.deb", quiet=True)
run("apt-get install -y -qq /content/rustdesk.deb")
print("   Verifying...")
run("which rustdesk")
run("touch /usr/lib/x86_64-linux-gnu/libcuda.so.1 && ldconfig 2>/dev/null", quiet=True)

# 3. Chrome
if INSTALL_CHROME:
    print("🌐 [3/6] Installing Google Chrome...")
    run("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /content/chrome.deb", quiet=True)
    run("apt-get install -y -qq /content/chrome.deb")
    run("sed -i 's|^Exec=/usr/bin/google-chrome-stable|Exec=/usr/bin/google-chrome-stable --no-sandbox --disable-dev-shm-usage --disable-gpu|' /usr/share/applications/google-chrome.desktop", quiet=True)
    run("update-desktop-database -q", quiet=True)

# 4. cleanup + start desktop
print("🧹 [4/6] Starting virtual display + XFCE...")
run("pkill -9 Xvfb; pkill -9 rustdesk; pkill -9 xfce4; sleep 2", quiet=True)

bg("Xvfb :99 -screen 0 1280x720x24", "/content/xvfb.log")
time.sleep(3)
bg("dbus-daemon --system --fork", "/content/dbus.log")
time.sleep(2)

env = {"DISPLAY": ":99"}
bg("startxfce4", "/content/xfce.log", env)
time.sleep(10)

# 5. RustDesk server
print("🔑 [5/6] Starting RustDesk...")
bg("rustdesk --server", "/content/rd_server.log", env)
time.sleep(8)
run(f"rustdesk --password {RUSTDESK_PASSWORD}", quiet=True)
bg("rustdesk --headless", "/content/rd_headless.log", env)
time.sleep(3)

# 6. get ID (poll)
print("⏳ [6/6] Waiting for RustDesk to register...")
rd_id = ""
for i in range(30):
    time.sleep(2)
    r = subprocess.run("rustdesk --get-id", shell=True, capture_output=True, text=True)
    rd_id = r.stdout.strip()
    if rd_id:
        print(f"   ✅ Registered after {i*2}s")
        break
    print(f"   ... waiting ({i*2}s)")

# keep-alive
display(IPython.display.Javascript('''
setInterval(function(){
  var b = document.querySelector("colab-connect-button");
  if (b) b.click();
  var ok = document.getElementById("ok");
  if (ok) ok.click();
}, 60000);
'''))

print("\n" + "="*55)
print("✅  YOUR COLAB DESKTOP IS READY")
print("="*55)
print(f"  RustDesk ID :  {rd_id if rd_id else '❌ FAILED - check logs'}")
print(f"  Password    :  {RUSTDESK_PASSWORD}")
print("="*55)
print("\n📱 Connect with RustDesk app using the ID above.")
print("⚠️  Keep this tab open. Session resets in ~9-12 hours.")